### Develop a Spark Structured Streaming application to process data from a selected data source. Implement suitable fault-tolerance mechanisms and checkpointing to ensure reliable data processing and recovery in case of failures.

In [2]:
!pip install pyspark

In [3]:
import os
import json
import time
import shutil

from pyspark.sql import SparkSession
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("StructuredStreamingDemo") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark Session Started Successfully!")

Spark Session Started Successfully!


In [4]:
input_path = "/content/spark_stream_data"
checkpoint_path = "/content/spark_stream_checkpoint"

shutil.rmtree(input_path, ignore_errors=True)
shutil.rmtree(checkpoint_path, ignore_errors=True)

os.makedirs(input_path, exist_ok=True)
os.makedirs(checkpoint_path, exist_ok=True)

print("Folders Created Successfully!")

Folders Created Successfully!


In [5]:
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("subject", StringType(), True),
    StructField("marks", IntegerType(), True),
    StructField("city", StringType(), True)
])

print("Student Record Schema Defined Successfully!")

Schema Defined Successfully!


In [6]:
stream_df = spark.readStream \
    .schema(schema) \
    .json(input_path)

print("JSON Streaming Data Source Created!")

Streaming Data Source Created!


In [7]:
filtered_stream = stream_df.dropDuplicates(["id"])

print("Duplicate Student IDs Will Be Removed!")

Duplicate Detection Applied!


In [8]:
streaming_query = filtered_stream.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("checkpointLocation", checkpoint_path) \
    .trigger(processingTime="2 seconds") \
    .start()

print("Streaming Started Successfully!")

Streaming Started Successfully!


In [9]:
data1 = [
    {"id": 1, "name": "Meera", "department": "Computer Science", "subject": "Big Data", "marks": 86, "city": "Gangtok"},
    {"id": 2, "name": "Arjun", "department": "Information Technology", "subject": "DBMS", "marks": 91, "city": "Siliguri"},
    {"id": 3, "name": "Kavya", "department": "AI and Data Science", "subject": "Machine Learning", "marks": 89, "city": "Namchi"},
    {"id": 4, "name": "Rohan", "department": "Computer Science", "subject": "Computer Networks", "marks": 78, "city": "Kalimpong"}
]

with open(input_path + "/batch1.json", "w") as f:
    for row in data1:
        f.write(json.dumps(row) + "\n")

time.sleep(5)
print("First Batch Sent Successfully!")

First Batch Sent Successfully!


In [10]:
data2 = [
    {"id": 5, "name": "Ananya", "department": "Electronics", "subject": "IoT", "marks": 94, "city": "Kolkata"},
    {"id": 6, "name": "Dev", "department": "AI and Data Science", "subject": "Python", "marks": 82, "city": "Jorethang"},
    {"id": 7, "name": "Ishita", "department": "Information Technology", "subject": "Cyber Security", "marks": 88, "city": "Darjeeling"},
    {"id": 2, "name": "Arjun", "department": "Information Technology", "subject": "DBMS", "marks": 91, "city": "Siliguri"}
]

with open(input_path + "/batch2.json", "w") as f:
    for row in data2:
        f.write(json.dumps(row) + "\n")

time.sleep(5)
print("Second Batch Sent Successfully! The repeated ID 2 is included to test duplicate removal.")

Second Batch Sent Successfully!


In [ ]:
data3 = [
    {"id": 8, "name": "Rahul", "department": "Mechanical Engineering", "subject": "Engineering Mechanics", "marks": 73, "city": "Pakyong"},
    {"id": 9, "name": "Sana", "department": "Computer Science", "subject": "Java", "marks": 96, "city": "Imphal"},
    {"id": 10, "name": "Nikhil", "department": "Electronics", "subject": "Digital Electronics", "marks": 81, "city": "Shillong"},
    {"id": 11, "name": "Priya", "department": "AI and Data Science", "subject": "Statistics", "marks": 90, "city": "Aizawl"}
]

with open(input_path + "/batch3.json", "w") as f:
    for row in data3:
        f.write(json.dumps(row) + "\n")

time.sleep(5)
print("Third Batch Sent Successfully!")

In [ ]:
data4 = [
    {"id": 12, "name": "Kabir", "department": "Information Technology", "subject": "Cloud Computing", "marks": 85, "city": "Kohima"},
    {"id": 13, "name": "Tanya", "department": "Computer Science", "subject": "Data Structures", "marks": 92, "city": "Agartala"},
    {"id": 14, "name": "Aman", "department": "Mechanical Engineering", "subject": "Thermodynamics", "marks": 76, "city": "Itanagar"},
    {"id": 15, "name": "Diya", "department": "AI and Data Science", "subject": "Deep Learning", "marks": 98, "city": "Guwahati"}
]

with open(input_path + "/batch4.json", "w") as f:
    for row in data4:
        f.write(json.dumps(row) + "\n")

time.sleep(5)
print("Fourth Batch Sent Successfully!")

In [11]:
streaming_query.stop()

print("Streaming Stopped Successfully!")

Streaming Stopped Successfully!


In [12]:
student_ids = [101, 102, 103, 104, 102, 105, 106, 103, 107, 108, 101]

repeated_ids = []

for student_id in student_ids:
    if student_ids.count(student_id) > 1 and student_id not in repeated_ids:
        repeated_ids.append(student_id)

print("Student IDs:", student_ids)
print("Repeated IDs:", repeated_ids)

Original Data: [10, 20, 30, 20, 40, 10, 50]
Duplicate Elements: [10, 20]


In [13]:
spark.stop()

print("Spark Session Closed Successfully!")

Spark Session Closed Successfully!
